In [4]:
import pandas as pd
import numpy as np
import math
import statistics as sts
import pandas as pd
from scipy.stats import trim_mean, iqr, median_abs_deviation

In [5]:
def determinar_clases_sturges(df, coluna):
    """
    Entrada: iniciar colocando uma coluna pandas
    saida: retorna os valores da amplitude, classes(k) e amplitude do intervalo (h)
    """
    coluna = sorted(df[coluna].dropna())
    amplitude_total = max(coluna) - min(coluna)
    k = round(np.sqrt(len(coluna)))
    h = amplitude_total/k
    return print(f" Amplitude Total: {amplitude_total}, Numero de Classes: {k}, Amplitude do Intervalo: {h}")

In [6]:
def determinar_clases_sturges_volumetrico(df, coluna):
    """
    Entrada: iniciar colocando uma coluna pandas para dados volumetricos
    saida: retorna os valores da amplitude, classes(k) e amplitude do intervalo (h)
    -- grande massa de dados.
    """
    n = len(coluna)
    coluna = sorted(df[coluna].dropna())
    amplitude_total = max(coluna) - min(coluna)
    k = 1 + 3.33 * math.log10(len(coluna))
    h = amplitude_total/k
    return print(f" Amplitude Total: {amplitude_total}, Numero de Classes: {k}, Amplitude do Intervalo: {h}")

In [50]:
def tabulacao_categoricos(df, coluna, dropna= True):
    """
    Entrada: dataframe pandas, nome da coluna entre aspas, se vc quiser os valores de NA, deixa o ultimo parametro True, senão, False.
    retorno: uma tabela de frequencias, com a frequencia, absoluta, relativa, percentil, acumulada e acumulada relativa
    """

    tabulacao_categoricos = pd.DataFrame(data = df[coluna].value_counts(dropna = dropna)).reset_index().rename(columns = {coluna:'Categoria', 'count': 'Frequencia_abs'})
    tabulacao_categoricos['frequencia_relativa'] = [i/tabulacao_categoricos['Frequencia_abs'].sum() for i in tabulacao_categoricos['Frequencia_abs']]
    tabulacao_categoricos['frequencia_percetual'] = [round(i*100, 2) for i in tabulacao_categoricos['frequencia_relativa']]
    var = 0
    feature = []
    for i in tabulacao_categoricos['Frequencia_abs']:
        var += i
        feature.append(var)
    tabulacao_categoricos['frequencia_acumulada'] = feature
    tabulacao_categoricos['frequencia_acumulada_relativa'] = [i/tabulacao_categoricos['Frequencia_abs'].sum() for i in tabulacao_categoricos['frequencia_acumulada']]
    return tabulacao_categoricos

In [8]:
def tabulacao_numericas(df, coluna, h):
    """
    Entrada: Dataframe Pandas
    Retorno: Retorna uma tabela analitica trazendo limite_inferior, limite_superior, frequencia_abs, frequencia_relativa, frequencia_percentual, frequencia_acumulada, frequencia_acumulada_relativa
    """
    dados = sorted(df[coluna].dropna())
    limite_inferior = min(dados)
    limite_superior = limite_inferior + h
    lista_apoio = []
    while limite_superior <= max(dados):
        slicing = [x for x  in dados if x < limite_superior]
        f_abs = len(slicing)
        intervalo = [limite_inferior, limite_superior, f_abs]
        lista_apoio.append(intervalo)
        limite_inferior = limite_superior + 1
        limite_superior += h
    tabela_analise = pd.DataFrame(data= lista_apoio, columns=['limite_inferior', 'limite_superior', 'frequencia_abs'])
    tabela_analise['frequencia_relativa'] = [i/tabela_analise['frequencia_abs'].sum() for i in tabela_analise['frequencia_abs']]
    tabela_analise['frequencia_percentual'] = [round(i*100, 2) for i in tabela_analise['frequencia_relativa']]
    var = 0
    feature = []
    for i in tabela_analise['frequencia_abs']:
        var += i
        feature.append(var)
    tabela_analise['frequencia_acumulada'] = feature
    tabela_analise['frequencia_acumulada_relativa'] = [i/tabela_analise['frequencia_abs'].sum() for i in tabela_analise['frequencia_acumulada']]
    return tabela_analise

In [9]:
def resumo_estatistico_media(df, proportiontocut= 0.05, axis=0, center= np.median, scale=1.0, nan_policy='omit', interpolation='linear'):
    """
    entrada: Dataframe pandas, não necessita filtrar as variáveis categóricas, o código já filtra
    desvio - diferença entre o valor observado e a media
    variancia - distancia entre o valor observado e a media
    desvio Padrão - indica o quanto os valores observados estão uniformes ou não. Ou seja, quanto maior o DP, mais o conjunto está disperso ok
    desvio medio absoluto - media dos desvios -  é um erro estatístico que calcula a média de distância entre cada par de pontos de dados reais e ajustados. ok
    desvio medio absoluto da mediana - não é influenciado por outliers, 
    intervalo - 
    percentis - 
    intervalos interquartis - 
    retorno: resumo estatístico de tendencias centrais para variáveis numéricas
    obs: nessa versão foi retirada o desvio abs médio pois a função no pandas foi depreciada
    """
    df = df.select_dtypes(exclude=['object'])
    df = df.fillna(value=0)
    media = df.mean()
    mediana = df.median()
    variancia = df.var()
    desvio_padrão = df.std()
    #media_abs_desvio = df.mad()
    colunas = df.columns 
    media_truncada = []
    desvio_abs_mediana = []
    iqr_list = []
    #media_abs_desvio = []
    intervalo = []
    for columns in df.columns:
        df1 = np.array(df[columns])
        #intervalo = max(df1) - min(df1)
        media_trunc = trim_mean(df1, proportiontocut= proportiontocut, axis=axis) #parametro da função
        #media_abs_desvio = (df1 - df1.mean()).absolute().mean()
        desv_abs_mediana = median_abs_deviation(df1, axis=axis, center=center, scale=scale, nan_policy= nan_policy)
        iqr_value = iqr(df1, axis=None, rng=(25, 75), scale=1.0, nan_policy= nan_policy, interpolation= interpolation, keepdims=False)
        #intervalo.append(intervalo)
        media_truncada.append(media_trunc)
        #media_abs_desvio.append(media_abs_desvio)
        desvio_abs_mediana.append(desv_abs_mediana)
        iqr_list.append(iqr_value)
    #intervalo_df = pd.DataFrame(intervalo, index=colunas, columns=['intervalo'])
    medias_truncadas_df = pd.DataFrame(media_truncada, index=colunas, columns=['media_truncada'])
    desvio_abs_mediana_df = pd.DataFrame(desvio_abs_mediana, index=colunas, columns=['desvio_abs_mediana'])
    iqr_df = pd.DataFrame(iqr_list, index=colunas, columns=['iqr'])
    mediadf =pd.DataFrame(media, columns=['media'])
    medianadf =pd.DataFrame(mediana, columns=['mediana'])
    variancia_df = pd.DataFrame(variancia, columns=['variancia'])
    desvio_padrão_df = pd.DataFrame(desvio_padrão, columns=['desvio_padrão'])
    #media_abs_desvio_df = pd.DataFrame(media_abs_desvio, columns=['media_abs_desvio'])
    resumo = mediadf.join(medias_truncadas_df)
    #resumo = resumo.join(intervalo_df)
    resumo = resumo.join(medianadf)
    resumo = resumo.join(variancia_df)
    resumo = resumo.join(desvio_padrão_df)
    #resumo = resumo.join(media_abs_desvio_df)
    resumo = resumo.join(desvio_abs_mediana_df)
    resumo = resumo.join(iqr_df)
    return resumo

# Dataset de Teste

In [10]:
df = pd.read_excel('C:/Users/nivea/Documents/developer/ciencia_de_dados/Datasets_estudos/Earlwood_Air_Data_17_18.xls')

In [11]:
display(df)

,Date,Time,EARLWOOD WDR 1h average [°],EARLWOOD TEMP 1h average [°C],EARLWOOD WSP 1h average [m/s],EARLWOOD NO 1h average [pphm],EARLWOOD NO2 1h average [pphm],EARLWOOD CO 1h average [ppm],EARLWOOD OZONE 1h average [pphm],EARLWOOD OZONE 4h rolling average [pphm],EARLWOOD PM10 1h average [µg/m³],EARLWOOD PM2.5 1h average [µg/m³],EARLWOOD HUMID 1h average [%],EARLWOOD SD1 1h average [°],EARLWOOD CO 8h rolling average [ppm]
0,01/01/2017,01:00,152.3,22.6,0.4,0.0,0.4,NaN,2.0,2.1,23.6,7.0,87.2,49.01,NaN
1,01/01/2017,02:00,134.0,22.6,0.3,NaN,NaN,NaN,NaN,2.2,21.0,6.6,87.2,46.56,NaN
2,01/01/2017,03:00,132.2,22.6,0.3,0.0,0.6,NaN,1.7,2.0,20.0,7.2,87.0,47.40,NaN
3,01/01/2017,04:00,125.8,22.7,0.2,0.0,0.5,NaN,1.7,1.8,21.4,7.1,87.2,53.70,NaN
4,01/01/2017,05:00,107.9,22.8,0.6,0.0,0.3,NaN,2.1,1.8,21.5,4.3,86.8,41.65,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8779,01/01/2018,20:00,164.8,23.0,0.9,0.0,0.2,NaN,2.9,2.8,23.7,8.5,78.6,48.50,NaN
8780,01/01/2018,21:00,181.5,22.8,1.0,0.0,0.1,NaN,2.9,2.9,18.6,10.8,78.1,44.08,NaN
8781,01/01/2018,22:00,197.8,22.9,1.1,0.0,0.1,NaN,3.0,2.9,14.4,8.2,76.0,38.30,NaN
8782,01/01/2018,23:00,201.4,22.6,1.1,0.0,0.1,NaN,3.0,3.0,13.5,5.5,75.3,37.33,NaN


In [12]:
df.columns

Index(['Date', 'Time', 'EARLWOOD WDR 1h average [°]',
       'EARLWOOD TEMP 1h average [°C]', 'EARLWOOD WSP 1h average [m/s]',
       'EARLWOOD NO 1h average [pphm]', 'EARLWOOD NO2 1h average [pphm]',
       'EARLWOOD CO 1h average [ppm]', 'EARLWOOD OZONE 1h average [pphm]',
       'EARLWOOD OZONE 4h rolling average [pphm]',
       'EARLWOOD PM10 1h average [µg/m³]', 'EARLWOOD PM2.5 1h average [µg/m³]',
       'EARLWOOD HUMID 1h average [%]', 'EARLWOOD SD1 1h average [°]',
       'EARLWOOD CO 8h rolling average [ppm]'],
      dtype='object')

# Funções em execução

In [13]:
determinar_clases_sturges(df, 'EARLWOOD WDR 1h average [°]')

 Amplitude Total: 359.9, Numero de Classes: 94, Amplitude do Intervalo: 3.828723404255319


In [14]:
tabulacao_numericas(df, 'EARLWOOD WDR 1h average [°]', 3.828723404255319)

,limite_inferior,limite_superior,frequencia_abs,frequencia_relativa,frequencia_percentual,frequencia_acumulada,frequencia_acumulada_relativa
0,0.100000,3.928723,52,0.000133,0.01,52,0.000133
1,4.928723,7.757447,105,0.000268,0.03,157,0.000401
2,8.757447,11.586170,169,0.000432,0.04,326,0.000833
3,12.586170,15.414894,250,0.000639,0.06,576,0.001472
4,16.414894,19.243617,325,0.000831,0.08,901,0.002303
...,...,...,...,...,...,...,...
89,341.856383,344.685106,8565,0.021895,2.19,356470,0.911245
90,345.685106,348.513830,8610,0.022010,2.20,365080,0.933255
91,349.513830,352.342553,8651,0.022115,2.21,373731,0.955370
92,353.342553,356.171277,8701,0.022242,2.22,382432,0.977612


In [15]:
resumo_estatistico_media(df)

,media,media_truncada,mediana,variancia,desvio_padrão,desvio_abs_mediana,iqr
EARLWOOD WDR 1h average [°],190.370162,191.806236,201.15,10307.330694,101.525025,87.65,185.925
EARLWOOD TEMP 1h average [°C],18.170708,18.150139,18.60,35.490290,5.957373,4.00,8.300
EARLWOOD WSP 1h average [m/s],1.251138,1.181900,1.10,0.840550,0.916815,0.60,1.300
EARLWOOD NO 1h average [pphm],1.216302,0.782241,0.10,6.619470,2.572833,0.10,0.900
EARLWOOD NO2 1h average [pphm],1.020184,0.967202,0.80,0.660245,0.812555,0.50,1.200
EARLWOOD CO 1h average [ppm],0.000000,0.000000,0.00,0.000000,0.000000,0.00,0.000
EARLWOOD OZONE 1h average [pphm],1.472199,1.393069,1.50,1.573823,1.254521,1.10,2.200
EARLWOOD OZONE 4h rolling average [pphm],1.505430,1.439021,1.50,1.363153,1.167541,0.90,1.900
EARLWOOD PM10 1h average [µg/m³],17.849852,17.143157,16.00,110.910072,10.531385,5.60,11.600
EARLWOOD PM2.5 1h average [µg/m³],7.154076,6.483279,5.80,44.624818,6.680181,2.90,6.000


# Variáveis Categóricas

In [18]:
df_iris = pd.read_csv('C:/Users/nivea/Documents/developer/ciencia_de_dados/Datasets_estudos/Iris.csv')

In [19]:
display(df_iris)

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa
...,...,...,...,...,...,...
145,146,6.7,3.0,5.2,2.3,Iris-virginica
146,147,6.3,2.5,5.0,1.9,Iris-virginica
147,148,6.5,3.0,5.2,2.0,Iris-virginica
148,149,6.2,3.4,5.4,2.3,Iris-virginica


In [52]:
tabulacao_categoricos(df_iris, 'Species', dropna= True)

,Categoria,Frequencia_abs,frequencia_relativa,frequencia_percetual,frequencia_acumulada,frequencia_acumulada_relativa
0,Iris-setosa,50,0.333333,33.33,50,0.333333
1,Iris-versicolor,50,0.333333,33.33,100,0.666667
2,Iris-virginica,50,0.333333,33.33,150,1.000000
